In [1]:
import torch
import json
import torch.nn.functional as F

In [24]:
def load_data(pt_path, json_path):
    """Wczytuje wektory z sieci i oryginalny przepis na dataset."""
    pt_data = torch.load(pt_path, map_location='cpu')
    features_tensor = pt_data['features']
    
    with open(json_path, 'r') as f:
        recipe = json.load(f)
        
    return features_tensor, recipe

def compute_sv_patcav(feature_name, value_0, value_1, features_tensor, json_recipe):
    """
    Różnica średnich (PATCAV). 
    Wylicza czysty wektor cechy pozbawiony szumu i tła.
    """
    idx_0 = [i for i, bird in enumerate(json_recipe) if bird.get(feature_name) == value_0]
    idx_1 = [i for i, bird in enumerate(json_recipe) if bird.get(feature_name) == value_1]
    
    if not idx_0 or not idx_1:
        print(f"BŁĄD: Brakuje ptaków dla {feature_name} ({value_0} lub {value_1})")
        return None
        
    mean_0 = features_tensor[idx_0].mean(dim=0)
    mean_1 = features_tensor[idx_1].mean(dim=0)
    
    return mean_1 - mean_0

def test_publications(pt_path, json_path, feature_A, values_A, feature_B, values_B):
    """
    Uniwersalna funkcja do testowania hipotez na dowolnych cechach i zbiorach.
    values_A i values_B to listy z dwiema wartościami do odjęcia, np. ['beak01.glb', 'beak02.glb']
    """
    features_tensor, json_recipe = load_data(pt_path, json_path)
    
    print(f"\n=======================================================")
    print(f"ANALIZA: Plik '{pt_path}'")
    print(f"Cecha A: {feature_A} (Przejście: {values_A[0]} -> {values_A[1]})")
    print(f"Cecha B: {feature_B} (Przejście: {values_B[0]} -> {values_B[1]})")
    
    sv_A = compute_sv_patcav(feature_A, values_A[0], values_A[1], features_tensor, json_recipe)
    sv_B = compute_sv_patcav(feature_B, values_B[0], values_B[1], features_tensor, json_recipe)
    
    if sv_A is None or sv_B is None: return
    
    sv_A = sv_A.unsqueeze(0)
    sv_B = sv_B.unsqueeze(0)
    
    pahde_cos_sim = F.cosine_similarity(sv_A, sv_B).item()
    
    diff_A_B = sv_A - sv_B
    hcep_cos_sim = F.cosine_similarity(diff_A_B, sv_A).item()
    
    print(f"-------------------------------------------------------")
    print(f"Pahde: SV(A) ⊥ SV(B)")
    print(f"Wynik Cosine Similarity: {pahde_cos_sim:7.4f}")
    
    print(f"-------------------------------------------------------")
    print(f"HCEP: SV(A) - SV(B) ⊥ SV(A)")
    print(f"Wynik Cosine Similarity: {hcep_cos_sim:7.4f}")
    print(f"=======================================================\n")

In [23]:
test_publications(
    pt_path='model_nzal_features.pt', 
    json_path='./testy_nzal/FunnyBirds/dataset_train.json',
    feature_A='beak_model', values_A=['beak01.glb', 'beak02.glb'],
    feature_B='foot_model', values_B=['foot01.glb', 'foot02.glb']
)


ANALIZA: Plik 'model_nzal_features.pt'
Cecha A: beak_model (Przejście: beak01.glb -> beak02.glb)
Cecha B: foot_model (Przejście: foot01.glb -> foot02.glb)
-------------------------------------------------------
Pahde: SV(A) ⊥ SV(B)
Wynik Cosine Similarity:  0.0392
-------------------------------------------------------
HCEP: SV(A) - SV(B) ⊥ SV(A)
Wynik Cosine Similarity:  0.6778



In [22]:
test_publications(
    pt_path='model_hier_features.pt', 
    json_path='./testy_hier/FunnyBirds/dataset_train.json',
    feature_A='beak_model', values_A=['beak01.glb', 'beak02.glb'],
    feature_B='foot_model', values_B=['foot01.glb', 'foot02.glb']
)


ANALIZA: Plik 'model_hier_features.pt'
Cecha A: beak_model (Przejście: beak01.glb -> beak02.glb)
Cecha B: foot_model (Przejście: foot01.glb -> foot02.glb)
-------------------------------------------------------
Pahde: SV(A) ⊥ SV(B)
Wynik Cosine Similarity:  0.6075
-------------------------------------------------------
HCEP: SV(A) - SV(B) ⊥ SV(A)
Wynik Cosine Similarity:  0.3944

